# Coding LLM Pretraining Dataset Builder

Builds a ~60-70 GB (raw UTF-8 text) pretraining corpus for a 1-5B parameter coding model and pushes it as sharded zstd Parquet to a **private Hugging Face Hub dataset repo**.

**Sources & raw-text byte budgets**

| Source | HF dataset | Budget |
| --- | --- | --- |
| Code | `bigcode/starcoderdata` (per language) | ~30 GB |
| Web (edu) | `HuggingFaceFW/fineweb-edu` `sample-10BT` | ~15 GB |
| Docs | StarCoder2 docs + GoodDocs + DevKB + Stack Exchange | ~10 GB |
| Math | `HuggingFaceTB/finemath` `finemath-4plus` | ~3 GB |
| Wikipedia | `wikimedia/wikipedia` `20231101.en` | ~2 GB |

**Design principles**

- **Streaming everywhere** (`streaming=True`): shards are written to local scratch, uploaded to the Hub, then deleted. Nothing large stays on Colab disk.
- **Resumable**: a JSON manifest on the Hub records completed shards. Re-running any cell skips finished work, so a Colab disconnect is not fatal.
- **Config-driven**: all knobs live in the `CONFIG` dict in the Setup cell.
- **Deterministic**: fixed seeds throughout.

Run the cells top to bottom. Each source cell is independent and idempotent. Training is intentionally out of scope for this notebook.


## 1. Install dependencies

Pinned installs so the notebook is reproducible. Restart the runtime if Colab prompts you to after the install completes.

In [ ]:
import sys, subprocess

# Pinned dependency set. hf_transfer accelerates Hub uploads/downloads.
PACKAGES = [
    "datasets==3.2.0",
    "huggingface_hub[hf_transfer]==0.27.1",
    "pyarrow==18.1.0",
    "zstandard==0.23.0",
    "trafilatura==1.12.2",
    "fasttext-wheel==0.9.2",
    "datasketch==1.6.5",
    "orjson==3.10.13",
    "detect-secrets==1.5.0",
    "tqdm==4.67.1",
    "requests==2.32.3",
]


def pip_install(packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


pip_install(PACKAGES)
print("Dependencies installed. If Colab asks to restart the runtime, do so, then continue from cell 2.")

: 

## 2. Setup and `CONFIG`

Reads the HF token from Colab Secrets (key name `HF_TOKEN`), enables `hf_transfer`, creates the private dataset repo, and defines every tunable knob in a single `CONFIG` dict.

To set the secret: Colab left sidebar -> key icon -> add `HF_TOKEN` with a **write**-scoped token, and toggle "Notebook access" on.

In [ ]:
import os

# hf_transfer must be enabled before huggingface_hub is imported anywhere.
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger("pretrain-builder")


def _resolve_hf_token():
    """Prefer Colab Secrets, fall back to env var / cached login."""
    try:
        from google.colab import userdata  # type: ignore

        token = userdata.get("HF_TOKEN")
        if token:
            return token
    except Exception:
        pass
    return os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")


HF_TOKEN = _resolve_hf_token()
if not HF_TOKEN:
    raise RuntimeError(
        "No HF token found. Add a write-scoped token named HF_TOKEN in Colab Secrets "
        "(key icon in the left sidebar) or set the HF_TOKEN environment variable."
    )
os.environ["HF_TOKEN"] = HF_TOKEN

from huggingface_hub import HfApi, whoami

_HF_API = HfApi(token=HF_TOKEN)
_WHOAMI = whoami(token=HF_TOKEN)
HF_USERNAME = _WHOAMI["name"]
log.info("Authenticated to the Hub as: %s", HF_USERNAME)

GB = 1024 ** 3  # gibibyte; "GB" throughout the notebook means raw UTF-8 text bytes

CONFIG = {
    # ---- Hub destination -------------------------------------------------
    "hf": {
        # Change the dataset name if you like; it is created under your account.
        "repo_id": f"{HF_USERNAME}/coder-pretrain-60gb",
        "private": True,
    },
    # ---- Reproducibility & scratch --------------------------------------
    "seed": 1234,
    "scratch_dir": "/content/scratch",  # ephemeral local staging for shards
    # ---- Shard writer ----------------------------------------------------
    "shard": {
        "target_bytes": 256 * 1024 * 1024,  # ~256 MB raw text per shard
        "max_rows": 200_000,                 # also flush a shard after this many rows
        "compression": "zstd",
        "compression_level": 9,
    },
    # ---- Per-source configuration ---------------------------------------
    "sources": {
        "code": {
            "enabled": True,
            "hf_dataset": "bigcode/starcoderdata",
            # data_dir (folder name in the repo) -> raw-text byte budget
            "languages": {
                "python":     8 * GB,
                "javascript": 4 * GB,
                "typescript": 4 * GB,
                "java":       3 * GB,
                "cpp":        3 * GB,
                "go":         2 * GB,
                "rust":       2 * GB,
                "sql":        1 * GB,
                "shell":      1 * GB,
                "html":       1 * GB,
                "css":        1 * GB,
            },
            "filters": {
                "min_bytes": 20,
                "max_bytes": 1 * 1024 * 1024,     # drop files > 1 MB
                "max_line_bytes": 1000,           # reject very long lines (minified)
                "max_mean_line_bytes": 100,       # reject dense/generated files
                "min_alpha_frac": 0.25,           # fraction of alphabetic chars
                "max_non_alnum_frac": 0.5,        # reject symbol soup
                "autogen_markers": ["@generated", "autogenerated", "auto-generated",
                                     "do not edit", "code generated by"],
            },
        },
        "web": {
            "enabled": True,
            "hf_dataset": "HuggingFaceFW/fineweb-edu",
            "hf_config": "sample-10BT",
            "budget_bytes": 15 * GB,
            "min_int_score": 3,       # edu-quality threshold
            "min_chars": 200,
            "max_chars": 100_000,
            "min_lang_score": 0.65,   # English via FineWeb's language_score field
        },
        "math": {
            "enabled": True,
            "hf_dataset": "HuggingFaceTB/finemath",
            "hf_config": "finemath-4plus",
            "budget_bytes": 3 * GB,
            "min_chars": 200,
            "max_chars": 100_000,
        },
        "wiki": {
            "enabled": True,
            "hf_dataset": "wikimedia/wikipedia",
            "hf_config": "20231101.en",
            "budget_bytes": 2 * GB,
            "min_chars": 500,         # drop stubs
        },
        "docs": {
            "enabled": True,
            "budget_bytes": 10 * GB,
            "min_chars": 200,
            "max_chars": 200_000,
            "crawl_delay_s": 0.35,            # politeness delay between Tier-2 requests
            "max_pages_per_site": 8000,
            "request_timeout_s": 20,
            "user_agent": "coder-pretrain-docs-bot/1.0 (+research; contact via HF)",
        },
    },
    # ---- Global text quality filters (shared across web/docs/wiki) ------
    "quality": {
        "min_words": 20,
        "max_symbol_word_ratio": 0.5,     # Gopher-style
        "max_duplicate_line_frac": 0.30,  # reject boilerplate-heavy docs
        "min_mean_word_len": 2.0,
        "max_mean_word_len": 12.0,
    },
    # ---- Deduplication ---------------------------------------------------
    "dedup": {
        "num_perm": 128,
        "jaccard_threshold": 0.8,
        "shingle_size": 5,           # word-level shingles for MinHash
        "near_dedup_max_rows": 4_000_000,  # above this, a source uses exact-dedup only (RAM guard)
        "max_words_for_minhash": 2000,     # cap words hashed per doc for speed
        # Upstream text datasets are already near-deduped; skip MinHash-LSH for speed.
        "near_dedup_disabled_sources": ["web", "math", "wiki", "docs"],
    },
    # ---- Decontamination against eval benchmarks ------------------------
    "decontam": {
        "benchmarks": [
            {"dataset": "openai_humaneval", "config": None, "split": "test", "fields": ["prompt", "canonical_solution"]},
            {"dataset": "mbpp", "config": None, "split": "test", "fields": ["text", "code"]},
            {"dataset": "gsm8k", "config": "main", "split": "test", "fields": ["question", "answer"]},
            # MATH mirror — hendrycks/competition_math is no longer on the Hub
            {"dataset": "EleutherAI/hendrycks_math", "config": "algebra", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "counting_and_probability", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "geometry", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "intermediate_algebra", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "number_theory", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "prealgebra", "split": "test", "fields": ["problem", "solution"]},
            {"dataset": "EleutherAI/hendrycks_math", "config": "precalculus", "split": "test", "fields": ["problem", "solution"]},
        ],
        "ngram": 13,   # word-level n-gram size for overlap detection
        "max_words_scan": 2000,  # only scan first N words per doc (benchmark items are short)
    },
    # ---- Finalize --------------------------------------------------------
    "finalize": {
        "val_fraction": 0.001,     # small held-out validation split
        "final_shard_bytes": 384 * 1024 * 1024,
        "shuffle_buckets": 64,             # random partitions for a memory-safe global shuffle
        "bucket_flush_bytes": 32 * 1024 * 1024,  # per-bucket RAM buffer before flushing to disk
    },
}

os.makedirs(CONFIG["scratch_dir"], exist_ok=True)

# Create the destination dataset repo (idempotent).
_HF_API.create_repo(
    repo_id=CONFIG["hf"]["repo_id"],
    repo_type="dataset",
    private=CONFIG["hf"]["private"],
    exist_ok=True,
)
log.info("Destination dataset repo ready: https://huggingface.co/datasets/%s", CONFIG["hf"]["repo_id"])
log.info("Scratch dir: %s", CONFIG["scratch_dir"])

: 

## 3. Helpers

Shared building blocks used by every source cell:

- **Unified schema / record builder** - every document becomes `{id, text, source, meta}`.
- **`ShardWriter`** - buffers records and flushes ~256 MB zstd Parquet shards to local scratch, uploads each to the Hub, then deletes the local copy.
- **`upload_batch_with_retry`** - upload multiple files in one Hub commit (used by finalize).
- **Resumable manifest** - per-source JSON state stored on the Hub (`manifest/<source>.json`). Records examples consumed, bytes/rows written, next shard index, and a `done` flag so re-running a cell resumes instead of restarting.
- **`process_stream`** - generic budget-bounded, resumable driver over any streaming iterable.
- **Filters** - code heuristics, Gopher/C4 text-quality heuristics, fastText language ID, regex PII/secret redaction.

In [ ]:
import hashlib
import json
import re
import time

import orjson
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

# ------------------------------------------------------------------ schema ---
SCHEMA = pa.schema([
    ("id", pa.string()),      # sha1 of text, used for exact dedup
    ("text", pa.string()),    # the training text
    ("source", pa.string()),  # code | web | math | wiki | docs
    ("meta", pa.string()),    # orjson-encoded provenance dict
])


def text_id(text: str) -> str:
    return hashlib.sha1(text.encode("utf-8", "ignore")).hexdigest()


def make_record(text: str, source: str, meta: dict | None = None) -> dict:
    return {
        "id": text_id(text),
        "text": text,
        "source": source,
        "meta": orjson.dumps(meta or {}).decode("utf-8"),
    }


# ------------------------------------------------------- hub upload helpers ---
def _is_empty_hub_commit(exc: Exception) -> bool:
    msg = str(exc).lower()
    return "empty commit" in msg or "no files have been modified" in msg


def upload_with_retry(local_path: str, repo_path: str, retries: int = 5) -> None:
    delay = 5
    for attempt in range(1, retries + 1):
        try:
            _HF_API.upload_file(
                path_or_fileobj=local_path,
                path_in_repo=repo_path,
                repo_id=CONFIG["hf"]["repo_id"],
                repo_type="dataset",
            )
            return
        except Exception as exc:  # noqa: BLE001 - transient Hub/network errors
            if _is_empty_hub_commit(exc):
                return
            if attempt == retries:
                raise
            log.warning("Upload of %s failed (attempt %s/%s): %s. Retrying in %ss.",
                        repo_path, attempt, retries, exc, delay)
            time.sleep(delay)
            delay = min(delay * 2, 60)


def upload_batch_with_retry(pairs: list, message: str, retries: int = 5) -> None:
    """Upload multiple files in a single Hub commit. pairs: [(local_path, repo_path), ...]"""
    from huggingface_hub import CommitOperationAdd, create_commit

    if not pairs:
        return
    delay = 5
    for attempt in range(1, retries + 1):
        try:
            ops = [
                CommitOperationAdd(path_in_repo=repo, path_or_fileobj=local)
                for local, repo in pairs
            ]
            create_commit(
                repo_id=CONFIG["hf"]["repo_id"],
                repo_type="dataset",
                operations=ops,
                commit_message=message,
                token=HF_TOKEN,
            )
            return
        except Exception as exc:  # noqa: BLE001
            if _is_empty_hub_commit(exc):
                return
            if attempt == retries:
                raise
            log.warning("Batch upload failed (attempt %s/%s): %s. Retrying in %ss.",
                        attempt, retries, exc, delay)
            time.sleep(delay)
            delay = min(delay * 2, 120)


# --------------------------------------------------------- resumable state ---
MANIFEST_DIR = os.path.join(CONFIG["scratch_dir"], "manifest")
os.makedirs(MANIFEST_DIR, exist_ok=True)


def _default_unit() -> dict:
    return {
        "done": False,
        "examples_consumed": 0,
        "bytes_written": 0,
        "rows_written": 0,
        "next_shard_index": 0,
    }


def _default_state(source: str) -> dict:
    state = _default_unit()
    state["source"] = source
    state["sub"] = {}  # per-subset (e.g. per-language) units live here
    return state


def load_state(source: str) -> dict:
    """Load per-source manifest from the Hub, or a fresh default if absent."""
    remote = f"manifest/{source}.json"
    try:
        path = hf_hub_download(
            repo_id=CONFIG["hf"]["repo_id"],
            repo_type="dataset",
            filename=remote,
            token=HF_TOKEN,
            force_download=True,
        )
        with open(path, "rb") as fh:
            state = json.loads(fh.read())
        state.setdefault("sub", {})
        return state
    except Exception:  # noqa: BLE001 - file simply may not exist yet
        return _default_state(source)


def save_state(source: str, state: dict) -> None:
    local = os.path.join(MANIFEST_DIR, f"{source}.json")
    with open(local, "wb") as fh:
        fh.write(orjson.dumps(state))
    upload_with_retry(local, f"manifest/{source}.json")


# --------------------------------------------------------------- ShardWriter ---
class ShardWriter:
    """Buffers records and flushes compressed Parquet shards to the Hub."""

    def __init__(self, source: str, subset: str | None = None, start_index: int = 0):
        self.source = source
        self.subset = subset
        self.index = start_index
        self.buf: list[dict] = []
        self.buf_bytes = 0
        self.total_bytes = 0
        self.total_rows = 0
        self.on_flush = None  # callback() invoked after every successful shard upload
        cfg = CONFIG["shard"]
        self.target_bytes = cfg["target_bytes"]
        self.max_rows = cfg["max_rows"]
        self.compression = cfg["compression"]
        self.level = cfg["compression_level"]

    def add(self, record: dict) -> None:
        self.buf.append(record)
        nbytes = len(record["text"].encode("utf-8"))
        self.buf_bytes += nbytes
        self.total_bytes += nbytes
        self.total_rows += 1
        if self.buf_bytes >= self.target_bytes or len(self.buf) >= self.max_rows:
            self.flush()

    def flush(self) -> None:
        if not self.buf:
            return
        base = (f"{self.subset}-{self.index:05d}.parquet"
                if self.subset else f"{self.source}-{self.index:05d}.parquet")
        local_path = os.path.join(CONFIG["scratch_dir"], base)
        table = pa.Table.from_pylist(self.buf, schema=SCHEMA)
        pq.write_table(table, local_path,
                       compression=self.compression,
                       compression_level=self.level)
        upload_with_retry(local_path, f"data/{self.source}/{base}")
        try:
            os.remove(local_path)
        except OSError:
            pass
        self.index += 1
        self.buf = []
        self.buf_bytes = 0
        if self.on_flush:
            self.on_flush()


def process_stream(source, unit_state, budget_bytes, make_iterable, transform,
                   persist, subset=None, log_every=50_000):
    """Budget-bounded, resumable driver.

    make_iterable(skip_n) -> iterable of raw examples with skip_n already applied.
    transform(example)    -> record dict, or None to drop the example.
    persist()             -> save the parent source state to the Hub.
    """
    tag = f"{source}/{subset}" if subset else source
    if unit_state.get("done"):
        log.info("[%s] already complete (%.2f GB). Skipping.", tag, unit_state["bytes_written"] / GB)
        return unit_state

    skip_n = unit_state["examples_consumed"]
    writer = ShardWriter(source, subset=subset, start_index=unit_state["next_shard_index"])
    writer.total_bytes = unit_state["bytes_written"]
    writer.total_rows = unit_state["rows_written"]
    seen = [skip_n]

    def _sync():
        unit_state["next_shard_index"] = writer.index
        unit_state["examples_consumed"] = seen[0]
        unit_state["bytes_written"] = writer.total_bytes
        unit_state["rows_written"] = writer.total_rows

    def on_flush():
        _sync()
        persist()

    writer.on_flush = on_flush

    if skip_n:
        log.info("[%s] resuming: skipping %s already-consumed examples.", tag, skip_n)

    try:
        for example in make_iterable(skip_n):
            seen[0] += 1
            record = transform(example)
            if record is not None:
                writer.add(record)
            if writer.total_bytes >= budget_bytes:
                break
            if seen[0] % log_every == 0:
                log.info("[%s] seen=%s kept=%.2f GB (%s rows)",
                         tag, seen[0], writer.total_bytes / GB, writer.total_rows)
    finally:
        writer.flush()
        _sync()

    unit_state["done"] = writer.total_bytes >= budget_bytes
    persist()
    log.info("[%s] finished: %.2f GB, %s rows, %s shards (done=%s)",
             tag, writer.total_bytes / GB, writer.total_rows, writer.index, unit_state["done"])
    return unit_state


# ------------------------------------------------------------- code filters ---
# Character-ratio checks run on a small sample and use C-level regex counts
# instead of Python per-character loops (orders of magnitude faster).
_RE_ALPHA = re.compile(r"[A-Za-z]")
_RE_NON_ALNUM = re.compile(r"[^0-9A-Za-z\s]")


def _sample(text: str, n: int = 4000) -> str:
    return text if len(text) <= n else text[:n]


def code_ok(text: str, f: dict) -> bool:
    nbytes = len(text.encode("utf-8"))
    if nbytes < f["min_bytes"] or nbytes > f["max_bytes"]:
        return False
    lines = text.split("\n")
    if not lines:
        return False
    if max((len(ln) for ln in lines), default=0) > f["max_line_bytes"]:
        return False
    if (len(text) / len(lines)) > f["max_mean_line_bytes"]:
        return False
    s = _sample(text)
    slen = max(len(s), 1)
    if len(_RE_ALPHA.findall(s)) / slen < f["min_alpha_frac"]:
        return False
    if len(_RE_NON_ALNUM.findall(s)) / slen > f["max_non_alnum_frac"]:
        return False
    low = s.lower()
    if any(marker in low for marker in f["autogen_markers"]):
        return False
    return True


# ---------------------------------------------------------- text quality -----
def quality_ok(text: str, min_chars: int | None = None, max_chars: int | None = None) -> bool:
    q = CONFIG["quality"]
    n = len(text)
    if min_chars is not None and n < min_chars:
        return False
    if max_chars is not None and n > max_chars:
        return False
    words = text.split()
    if len(words) < q["min_words"]:
        return False
    mean_wl = sum(len(w) for w in words) / len(words)
    if mean_wl < q["min_mean_word_len"] or mean_wl > q["max_mean_word_len"]:
        return False
    symbol_ratio = (text.count("#") + text.count("...")) / len(words)
    if symbol_ratio > q["max_symbol_word_ratio"]:
        return False
    lines = [ln.strip() for ln in text.split("\n") if ln.strip()]
    if lines:
        dup_frac = 1.0 - (len(set(lines)) / len(lines))
        if dup_frac > q["max_duplicate_line_frac"]:
            return False
    return True


# --------------------------------------------------------- language id -------
_LID_MODEL = None


def _get_lid():
    global _LID_MODEL
    if _LID_MODEL is None:
        import fasttext
        model_path = hf_hub_download(
            "facebook/fasttext-language-identification", "model.bin", token=HF_TOKEN
        )
        _LID_MODEL = fasttext.load_model(model_path)
    return _LID_MODEL


def detect_lang(text: str) -> tuple[str, float]:
    model = _get_lid()
    sample = text[:2000].replace("\n", " ").strip()
    if not sample:
        return "unknown", 0.0
    # Use the C API directly — FastText.predict() breaks on NumPy 2.x
    # (np.array(..., copy=False) raises ValueError).
    predictions = model.f.predict(sample + "\n", 1)
    label = predictions[0][0]
    if isinstance(label, bytes):
        label = label.decode("utf-8")
    return label.replace("__label__", ""), float(predictions[0][1])


def is_english(text: str, min_score: float) -> bool:
    lang, score = detect_lang(text)
    return lang.split("_")[0] in ("eng", "en") and score >= min_score


# ------------------------------------------------------------- PII redaction --
_RE_EMAIL = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
_RE_IPV4 = re.compile(r"\b(?:\d{1,3}\.){3}\d{1,3}\b")
_RE_AWS = re.compile(r"\b(?:AKIA|ASIA)[0-9A-Z]{16}\b")
_RE_PRIVKEY = re.compile(r"-----BEGIN [A-Z ]*PRIVATE KEY-----[\s\S]*?-----END [A-Z ]*PRIVATE KEY-----")
_RE_TOKEN = re.compile(r"\b(?:ghp|gho|ghu|ghs|xox[baprs]|sk-)[A-Za-z0-9_\-]{16,}\b")


def redact_pii(text: str) -> str:
    text = _RE_PRIVKEY.sub("[PRIVATE_KEY]", text)
    text = _RE_AWS.sub("[AWS_KEY]", text)
    text = _RE_TOKEN.sub("[TOKEN]", text)
    text = _RE_EMAIL.sub("[EMAIL]", text)
    text = _RE_IPV4.sub("[IP]", text)
    return text


def list_source_shards(source: str) -> list[str]:
    """Repo-relative paths of all uploaded data shards for a source."""
    files = _HF_API.list_repo_files(repo_id=CONFIG["hf"]["repo_id"], repo_type="dataset")
    prefix = f"data/{source}/"
    return sorted(f for f in files if f.startswith(prefix) and f.endswith(".parquet"))


log.info("Helpers ready.")

## 4. Code source - `bigcode/starcoderdata`

Streams each configured language separately (`data_dir=<lang>`), applies code-quality heuristics, and fills each language's byte budget. Content is already near-deduplicated, decontaminated, and PII-scrubbed by BigCode, so we only apply lightweight structural filtering here.

**Gated dataset**: you must accept the terms once at https://huggingface.co/datasets/bigcode/starcoderdata while logged in with the same account as your `HF_TOKEN`, otherwise streaming will 403.

Per-language state lives under `manifest/code.json -> sub[<lang>]`, so a disconnect resumes at the language and offset where it stopped.

In [ ]:
from datasets import load_dataset


def build_code_source():
    cfg = CONFIG["sources"]["code"]
    if not cfg["enabled"]:
        log.info("Code source disabled; skipping.")
        return

    state = load_state("code")
    filters = cfg["filters"]

    def persist():
        save_state("code", state)

    for lang, budget in cfg["languages"].items():
        unit = state["sub"].get(lang) or _default_unit()
        state["sub"][lang] = unit

        def make_iterable(skip_n, _lang=lang):
            ds = load_dataset(cfg["hf_dataset"], data_dir=_lang, split="train",
                              streaming=True, token=HF_TOKEN)
            return ds.skip(skip_n) if skip_n else ds

        def transform(ex, _lang=lang):
            content = ex.get("content")
            if not content or not code_ok(content, filters):
                return None
            meta = {
                "language": _lang,
                "repo": ex.get("max_stars_repo_name"),
                "path": ex.get("max_stars_repo_path"),
                "stars": ex.get("max_stars_count"),
            }
            return make_record(content, "code", meta)

        process_stream("code", unit, budget, make_iterable, transform, persist, subset=lang)

    total = sum(u["bytes_written"] for u in state["sub"].values())
    log.info("=== Code source complete: %.2f GB across %s languages ===",
             total / GB, len(cfg["languages"]))


build_code_source()

README.md: 0.00B [00:00, ?B/s]

DatasetNotFoundError: Dataset 'bigcode/starcoderdata' is a gated dataset on the Hub. Visit the dataset page at https://huggingface.co/datasets/bigcode/starcoderdata to ask for access.

: 

## 5. Web (educational) - `HuggingFaceFW/fineweb-edu` `sample-10BT`

Keeps only `int_score >= 3` (educational quality), filters to English using the dataset's pre-computed `language` / `language_score` fields (no fastText download), enforces length + Gopher/C4 quality heuristics, and redacts PII. Fills a 15 GB budget.

In [ ]:
def build_web_source():
    cfg = CONFIG["sources"]["web"]
    if not cfg["enabled"]:
        log.info("Web source disabled; skipping.")
        return

    state = load_state("web")

    def persist():
        save_state("web", state)

    def make_iterable(skip_n):
        ds = load_dataset(cfg["hf_dataset"], name=cfg["hf_config"], split="train",
                          streaming=True, token=HF_TOKEN)
        return ds.skip(skip_n) if skip_n else ds

    def transform(ex):
        if int(ex.get("int_score", 0)) < cfg["min_int_score"]:
            return None
        # FineWeb-Edu already ships language + language_score — no fastText needed.
        lang = (ex.get("language") or "").lower()
        lang_score = float(ex.get("language_score") or 0.0)
        if lang not in ("en", "eng") or lang_score < cfg["min_lang_score"]:
            return None
        text = ex.get("text") or ""
        if not quality_ok(text, cfg["min_chars"], cfg["max_chars"]):
            return None
        text = redact_pii(text)
        meta = {
            "url": ex.get("url"),
            "score": ex.get("score"),
            "dump": ex.get("dump"),
            "language": lang,
            "language_score": lang_score,
        }
        return make_record(text, "web", meta)

    process_stream("web", state, cfg["budget_bytes"], make_iterable, transform, persist)
    log.info("=== Web source complete: %.2f GB ===", state["bytes_written"] / GB)


build_web_source()

## 6. Math - `HuggingFaceTB/finemath` `finemath-4plus`

The higher-quality FineMath subset (detailed step-by-step explanations, Markdown + LaTeX). Light length filtering only - we deliberately skip the symbol-ratio quality check and language ID because math content is symbol-dense and often not classified as plain English. Fills a 3 GB budget.

In [ ]:
def build_math_source():
    cfg = CONFIG["sources"]["math"]
    if not cfg["enabled"]:
        log.info("Math source disabled; skipping.")
        return

    state = load_state("math")

    def persist():
        save_state("math", state)

    def make_iterable(skip_n):
        ds = load_dataset(cfg["hf_dataset"], name=cfg["hf_config"], split="train",
                          streaming=True, token=HF_TOKEN)
        return ds.skip(skip_n) if skip_n else ds

    def transform(ex):
        text = ex.get("text") or ""
        n = len(text)
        if n < cfg["min_chars"] or n > cfg["max_chars"]:
            return None
        meta = {"url": ex.get("url"), "score": ex.get("score")}
        return make_record(text, "math", meta)

    process_stream("math", state, cfg["budget_bytes"], make_iterable, transform, persist)
    log.info("=== Math source complete: %.2f GB ===", state["bytes_written"] / GB)


build_math_source()

## 7. Wikipedia - `wikimedia/wikipedia` `20231101.en`

Cleaned English articles. We strip trailing boilerplate sections (References / See also / External links / Further reading / Notes / Bibliography) and drop stubs shorter than 500 characters. Fills a 2 GB budget.

In [ ]:
_WIKI_BOILERPLATE = {
    "references", "see also", "external links", "further reading",
    "notes", "bibliography", "citations", "sources", "footnotes",
}


def clean_wiki(text: str) -> str:
    lines = text.split("\n")
    cut = len(lines)
    for i, ln in enumerate(lines):
        if ln.strip().lower() in _WIKI_BOILERPLATE:
            cut = i
            break
    return "\n".join(lines[:cut]).strip()


def build_wiki_source():
    cfg = CONFIG["sources"]["wiki"]
    if not cfg["enabled"]:
        log.info("Wiki source disabled; skipping.")
        return

    state = load_state("wiki")

    def persist():
        save_state("wiki", state)

    def make_iterable(skip_n):
        ds = load_dataset(cfg["hf_dataset"], cfg["hf_config"], split="train",
                          streaming=True, token=HF_TOKEN)
        return ds.skip(skip_n) if skip_n else ds

    def transform(ex):
        text = clean_wiki(ex.get("text") or "")
        if len(text) < cfg["min_chars"]:
            return None
        meta = {"title": ex.get("title"), "url": ex.get("url"), "wiki_id": ex.get("id")}
        return make_record(text, "wiki", meta)

    process_stream("wiki", state, cfg["budget_bytes"], make_iterable, transform, persist)
    log.info("=== Wiki source complete: %.2f GB ===", state["bytes_written"] / GB)


build_wiki_source()

## 8. Documentation corpus (Hub-only)

No more git clones / HTML scraping. Docs are streamed from four Hugging Face datasets, in order, until the 10 GB budget is hit:

1. [`SivilTaram/starcoder2-documentation`](https://huggingface.co/datasets/SivilTaram/starcoder2-documentation) — StarCoder2 docs extras (Read the Docs / package managers / programming books)
2. [`MRiabov/gooddocs-v0`](https://huggingface.co/datasets/MRiabov/gooddocs-v0) — high-quality docs from top GitHub repos (content parquet only; avoids multi-schema CastError)
3. [`nuhmanpk/dev-knowledge-base`](https://huggingface.co/datasets/nuhmanpk/dev-knowledge-base) — official-style language/framework/tool docs
4. [`marin-community/stackexchange-markdown`](https://huggingface.co/datasets/marin-community/stackexchange-markdown) — **volume filler** (programming SE sites preferred)

`FORCE_RESET_DOCS = True` wipes previous docs shards/manifest before re-collecting. Set to `False` after a successful run so resumes work normally.


In [ ]:
from itertools import islice

from datasets import load_dataset

# Set True once to wipe the previous underfilled / mixed docs shards+manifest.
# Flip to False after a successful docs run so resumes work normally.
FORCE_RESET_DOCS = True

# Hub-only docs mix. Order matters: curated manuals first, Stack Exchange last
# so it only fills remaining budget up to CONFIG["sources"]["docs"]["budget_bytes"].
DOCS_HF_DATASETS = [
    {
        "name": "starcoder2-documentation",
        "hf_dataset": "SivilTaram/starcoder2-documentation",
        "text_field": "content",
    },
    {
        # GoodDocs has multiple parquet schemas in one repo — load the text file only.
        "name": "gooddocs",
        "hf_dataset": "MRiabov/gooddocs-v0",
        "text_field": "content",
        "data_files": [
            "hf://datasets/MRiabov/gooddocs-v0@main/filtered_texts.parquet",
            "hf://datasets/MRiabov/gooddocs-v0@main/cleaned_texts_on_metadata_only.parquet",
        ],
    },
    {
        "name": "dev-knowledge-base",
        "hf_dataset": "nuhmanpk/dev-knowledge-base",
        "text_field": "content",
    },
    {
        # Volume filler — last. Prefer programming-related SE sites when URL present.
        "name": "stackexchange",
        "hf_dataset": "marin-community/stackexchange-markdown",
        "text_field": "text",
        "url_prefer": [
            "stackoverflow.com",
            "serverfault.com",
            "superuser.com",
            "askubuntu.com",
            "unix.stackexchange.com",
            "softwareengineering.stackexchange.com",
            "codereview.stackexchange.com",
            "devops.stackexchange.com",
            "security.stackexchange.com",
            "dba.stackexchange.com",
            "cs.stackexchange.com",
            "datascience.stackexchange.com",
            "ai.stackexchange.com",
            "ethereum.stackexchange.com",
            "tex.stackexchange.com",
        ],
        "url_prefer_strict": False,
    },
]


def _load_docs_hf_stream(spec: dict):
    """Stream one Hub docs dataset. Supports optional explicit data_files."""
    try:
        if spec.get("data_files"):
            # Try candidates in order until one loads (GoodDocs multi-file case).
            last_err = None
            for path in spec["data_files"]:
                try:
                    return load_dataset(
                        "parquet",
                        data_files=path,
                        split="train",
                        streaming=True,
                        token=HF_TOKEN,
                    )
                except Exception as exc:  # noqa: BLE001
                    last_err = exc
                    log.warning("[docs] could not load %s (%s); trying next", path, exc)
            log.warning("[docs] all data_files failed for %s: %s",
                        spec["hf_dataset"], last_err)
            return None

        kwargs = dict(split="train", streaming=True, token=HF_TOKEN)
        name = spec.get("hf_config")
        if name:
            return load_dataset(spec["hf_dataset"], name=name, **kwargs)
        return load_dataset(spec["hf_dataset"], **kwargs)
    except Exception as exc:  # noqa: BLE001
        log.warning("[docs] load_dataset(%s) failed (%s); trying parquet glob fallback",
                    spec["hf_dataset"], exc)
        try:
            return load_dataset(
                "parquet",
                data_files=f"hf://datasets/{spec['hf_dataset']}@main/**/*.parquet",
                split="train",
                streaming=True,
                token=HF_TOKEN,
            )
        except Exception as exc2:  # noqa: BLE001
            log.warning("[docs] parquet fallback also failed for %s: %s",
                        spec["hf_dataset"], exc2)
            return None


def _url_preferred(url: str, prefer: list | None) -> bool:
    if not prefer:
        return True
    u = (url or "").lower()
    return any(p in u for p in prefer)


def docs_examples():
    """Stream Hub docs datasets in order until the outer budget is exhausted."""
    for spec in DOCS_HF_DATASETS:
        field = spec.get("text_field", "content")
        prefer = spec.get("url_prefer")
        strict = bool(spec.get("url_prefer_strict", False))
        ds = _load_docs_hf_stream(spec)
        if ds is None:
            continue
        log.info("[docs] streaming Hub dataset %s (text_field=%s)",
                 spec["hf_dataset"], field)
        kept = skipped = 0
        for ex in ds:
            text = ex.get(field) or ex.get("text") or ""
            if not text or not str(text).strip():
                continue
            text = str(text).strip()
            url = ex.get("url") or ""
            if prefer:
                ok = _url_preferred(url, prefer)
                if not ok and strict:
                    skipped += 1
                    continue
            meta = {
                "dataset": spec["name"],
                "hf_dataset": spec["hf_dataset"],
            }
            for k in ("owner", "repo", "file_rel_repo", "source", "url", "title",
                      "category", "language", "lang", "project", "id"):
                if ex.get(k) is not None:
                    meta[k] = ex[k]
            if prefer:
                meta["url_preferred"] = _url_preferred(url, prefer)
            kept += 1
            yield {"text": text, "meta": meta}
        log.info("[docs] %s: yielded %s nonempty docs (skipped_strict=%s)",
                 spec["name"], kept, skipped)


def _reset_docs_on_hub():
    """Delete previous docs shards + reset the Hub manifest."""
    from huggingface_hub import CommitOperationDelete

    repo = CONFIG["hf"]["repo_id"]
    files = _HF_API.list_repo_files(repo_id=repo, repo_type="dataset")
    to_delete = [f for f in files if f.startswith("data/docs/") and f.endswith(".parquet")]
    if "manifest/docs.json" in files:
        to_delete.append("manifest/docs.json")
    if not to_delete:
        log.info("[docs] nothing to reset on Hub")
        return
    log.info("[docs] resetting %s Hub files before re-collection", len(to_delete))
    batch = 50
    for i in range(0, len(to_delete), batch):
        chunk = to_delete[i:i + batch]
        try:
            ops = [CommitOperationDelete(path_in_repo=p) for p in chunk]
            _HF_API.create_commit(
                repo_id=repo,
                repo_type="dataset",
                operations=ops,
                commit_message=f"reset docs collection ({i // batch + 1})",
            )
        except Exception as exc:  # noqa: BLE001
            log.warning("[docs] batch delete failed (%s); falling back", exc)
            for p in chunk:
                try:
                    _HF_API.delete_file(path_in_repo=p, repo_id=repo, repo_type="dataset")
                except Exception as e2:  # noqa: BLE001
                    log.warning("[docs] could not delete %s: %s", p, e2)


def build_docs_source():
    cfg = CONFIG["sources"]["docs"]
    if not cfg["enabled"]:
        log.info("Docs source disabled; skipping.")
        return

    if FORCE_RESET_DOCS:
        _reset_docs_on_hub()
        log.info("[docs] FORCE_RESET_DOCS complete — starting Hub-only collection")

    state = load_state("docs")
    if FORCE_RESET_DOCS or (
        state.get("bytes_written", 0) < (cfg["budget_bytes"] // 10)
        and state.get("done")
    ):
        state = _default_state("docs")

    def persist():
        save_state("docs", state)

    def make_iterable(skip_n):
        return islice(docs_examples(), skip_n, None)

    def transform(ex):
        text = ex["text"]
        if not quality_ok(text, cfg["min_chars"], cfg["max_chars"]):
            return None
        return make_record(redact_pii(text), "docs", ex["meta"])

    process_stream("docs", state, cfg["budget_bytes"], make_iterable, transform, persist, log_every=5_000)
    log.info("=== Docs source complete: %.2f GB (done=%s) ===",
             state["bytes_written"] / GB, state.get("done"))
    if state["bytes_written"] < cfg["budget_bytes"] * 0.25:
        log.warning(
            "[docs] Only collected %.2f GB of the %.2f GB budget. "
            "Check Hub dataset yield logs above, or raise the Stack Exchange share.",
            state["bytes_written"] / GB, cfg["budget_bytes"] / GB,
        )


build_docs_source()


## 9. Global dedup and decontamination

Run once, after all source cells are done. Produces a `dedup/removed_ids.parquet` list of document ids to drop; `finalize` applies it. Nothing is deleted destructively here, so this cell is safe to re-run.

1. **Exact dedup** - the `id` column is a sha1 of the text, so identical documents collapse to one (global, cheap).
2. **Near-dedup** - per-source MinHash-LSH (128 perms, Jaccard 0.8) over word shingles. Disabled for `web`/`math`/`wiki`/`docs` (already near-deduped upstream) and for sources above `dedup.near_dedup_max_rows` (RAM guard).
3. **Decontamination** - drops any document whose first `decontam.max_words_scan` words contain a 13-gram from HumanEval, MBPP, GSM8K, or MATH test splits.

Text sources use exact-dedup + capped decontam scans for Colab-friendly runtime (~few hours total vs ~15+ hours with full near-dedup on web).


In [ ]:
import gc

from datasketch import MinHash, MinHashLSH
from datasets import load_dataset

SOURCES = ["code", "web", "math", "wiki", "docs"]


def _source_row_count(state: dict) -> int:
    """Manifest rows for a source; code stores per-language counts under sub."""
    if state.get("sub"):
        return sum(u.get("rows_written", 0) for u in state["sub"].values())
    return state.get("rows_written", 0)


def _minhash(text: str, num_perm: int, k: int, max_words: int) -> MinHash:
    m = MinHash(num_perm=num_perm)
    words = text.split()
    if len(words) > max_words:
        words = words[:max_words]
    if len(words) < k:
        shingles = {" ".join(words)} if words else {""}
    else:
        shingles = {" ".join(words[i:i + k]) for i in range(len(words) - k + 1)}
    for s in shingles:
        m.update(s.encode("utf-8"))
    return m


def _load_benchmark_dataset(spec: dict):
    kwargs = {"split": spec["split"], "token": HF_TOKEN}
    if spec["config"] is not None:
        return load_dataset(spec["dataset"], spec["config"], **kwargs)
    return load_dataset(spec["dataset"], **kwargs)


def _load_contam_ngrams() -> set:
    n = CONFIG["decontam"]["ngram"]
    ngrams: set = set()
    for b in CONFIG["decontam"]["benchmarks"]:
        try:
            ds = _load_benchmark_dataset(b)
        except Exception as exc:  # noqa: BLE001
            log.warning("Could not load benchmark %s: %s", b["dataset"], exc)
            continue
        for row in ds:
            for field in b["fields"]:
                words = (row.get(field) or "").split()
                for i in range(len(words) - n + 1):
                    ngrams.add(hash(" ".join(words[i:i + n])))
    log.info("Decontamination set: %s benchmark %s-grams", len(ngrams), n)
    return ngrams


def _is_contaminated(text: str, contam: set, n: int, max_words: int | None = None) -> bool:
    words = text.split()
    if max_words is not None and len(words) > max_words:
        words = words[:max_words]
    for i in range(len(words) - n + 1):
        if hash(" ".join(words[i:i + n])) in contam:
            return True
    return False


def run_dedup_and_decontam():
    dcfg = CONFIG["dedup"]
    n = CONFIG["decontam"]["ngram"]
    max_words_scan = CONFIG["decontam"].get("max_words_scan")
    near_disabled = set(dcfg.get("near_dedup_disabled_sources", []))
    contam = _load_contam_ngrams()

    seen_ids: set = set()
    removed: set = set()

    for source in SOURCES:
        shards = list_source_shards(source)
        if not shards:
            continue
        state = load_state(source)
        n_rows = _source_row_count(state)
        if source in near_disabled:
            near_enabled = False
            log.info("[dedup/%s] near-dedup disabled (upstream-deduped text source); exact-dedup only.", source)
        else:
            near_enabled = 0 < n_rows <= dcfg["near_dedup_max_rows"]
            if not near_enabled:
                if n_rows == 0:
                    log.warning("[dedup/%s] row count unknown; exact-dedup only.", source)
                else:
                    log.info(
                        "[dedup/%s] %s rows > near-dedup cap (%s); exact-dedup only.",
                        source,
                        n_rows,
                        dcfg["near_dedup_max_rows"],
                    )
        lsh = MinHashLSH(threshold=dcfg["jaccard_threshold"], num_perm=dcfg["num_perm"]) if near_enabled else None

        seen = exact_rm = near_rm = contam_rm = 0
        for shard in shards:
            path = hf_hub_download(repo_id=CONFIG["hf"]["repo_id"], repo_type="dataset",
                                   filename=shard, token=HF_TOKEN)
            table = pq.read_table(path, columns=["id", "text"])
            ids = table.column("id").to_pylist()
            texts = table.column("text").to_pylist()
            for _id, text in zip(ids, texts):
                seen += 1
                if _id in seen_ids:
                    removed.add(_id); exact_rm += 1; continue
                seen_ids.add(_id)
                if contam and _is_contaminated(text, contam, n, max_words_scan):
                    removed.add(_id); contam_rm += 1; continue
                if lsh is not None:
                    mh = _minhash(text, dcfg["num_perm"], dcfg["shingle_size"], dcfg["max_words_for_minhash"])
                    if lsh.query(mh):
                        removed.add(_id); near_rm += 1; continue
                    lsh.insert(_id, mh)
            try:
                os.remove(path)
            except OSError:
                pass
        log.info("[dedup/%s] seen=%s exact_dup=%s near_dup=%s contaminated=%s",
                 source, seen, exact_rm, near_rm, contam_rm)
        del lsh
        gc.collect()

    # Persist the removal list for finalize to apply.
    removed_table = pa.table({"id": pa.array(sorted(removed), type=pa.string())})
    local = os.path.join(CONFIG["scratch_dir"], "removed_ids.parquet")
    pq.write_table(removed_table, local, compression="zstd")
    upload_with_retry(local, "dedup/removed_ids.parquet")
    os.remove(local)
    log.info("=== Dedup + decontam complete: %s documents flagged for removal ===", len(removed))


run_dedup_and_decontam()


## 10. Finalize

Run after dedup. Resumable via `manifest/finalize.json` on the Hub.

1. **Mirror** — `snapshot_download` of `data/**` to local `data_mirror/` (skipped if already present).
2. **Phase 1 (Arrow)** — filter `removed_ids`, hash-partition into 64 on-disk buckets (one shard at a time, chunked).
3. **Phase 2** — shuffle each bucket part-by-part, train/val split, write `pending/{train,validation}/bucket_XXX/`.
4. **Phase 3** — merge pending bucket files into ~384 MB final shards locally.
5. **Batch upload** — push final shards in commits of `upload_batch_size` files (default 16).

Re-run this cell after a Colab disconnect; completed sources/buckets/uploads are skipped.


In [ ]:
import glob
from pathlib import Path
import random
import shutil
import zlib
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pyarrow.compute as pc
from huggingface_hub import snapshot_download

SOURCES = ["code", "web", "math", "wiki", "docs"]


def _fmt_gb(nbytes: int) -> str:
    return f"{nbytes / GB:.2f} GB"


def _default_finalize_state() -> dict:
    return {
        "mirror_complete": False,
        "partitioned_sources": [],
        "buckets_done": [],
        "per_source_final": {},
        "staged_train_parts": [],
        "staged_val_parts": [],
        "uploaded_train_parts": [],
        "uploaded_val_parts": [],
        "train_rows": 0,
        "train_bytes": 0,
        "val_rows": 0,
        "val_bytes": 0,
        "done": False,
    }


def load_finalize_state() -> dict:
    try:
        p = hf_hub_download(
            repo_id=CONFIG["hf"]["repo_id"],
            repo_type="dataset",
            filename="manifest/finalize.json",
            token=HF_TOKEN,
            force_download=True,
        )
        with open(p, "rb") as fh:
            state = json.loads(fh.read())
        for k, v in _default_finalize_state().items():
            state.setdefault(k, v)
        return state
    except Exception:  # noqa: BLE001
        return _default_finalize_state()


def save_finalize_state(state: dict) -> None:
    local = os.path.join(MANIFEST_DIR, "finalize.json")
    with open(local, "wb") as fh:
        fh.write(orjson.dumps(state))
    upload_with_retry(local, "manifest/finalize.json")


def _bucket_for_id(doc_id: str, k: int) -> int:
    return zlib.crc32(doc_id.encode()) % k


def _table_text_bytes(table: pa.Table) -> int:
    if table.num_rows == 0:
        return 0
    return int(pc.sum(pc.utf8_length(table["text"])).as_py() or 0)


def _write_dataset_card(stats: dict) -> str:
    fin = stats["final"]
    rows = [
        "| Source | Collected rows | Collected size | Final rows | Final size |",
        "| --- | --- | --- | --- | --- |",
    ]
    for source in SOURCES:
        col = stats["per_source_collected"].get(source, {"rows": 0, "bytes": 0})
        fn = stats["per_source_final"].get(source, {"rows": 0, "bytes": 0})
        rows.append(
            f"| {source} | {col['rows']:,} | {_fmt_gb(col['bytes'])} | "
            f"{fn['rows']:,} | {_fmt_gb(fn['bytes'])} |"
        )
    table = "\n".join(rows)
    return f"""---
license: other
language:
- en
size_categories:
- 10B<n<100B
task_categories:
- text-generation
tags:
- code
- pretraining
- the-stack
- fineweb-edu
configs:
- config_name: default
  data_files:
  - split: train
    path: final/train/*.parquet
  - split: validation
    path: final/validation/*.parquet
---

# Coding LLM Pretraining Corpus

A cleaned, deduplicated, and decontaminated pretraining corpus (~60-70 GB of raw UTF-8 text)
for training a 1-5B parameter coding model. Built with `build_pretrain_dataset.ipynb`.

## Composition

{table}

**Total final corpus:** {_fmt_gb(fin['train_bytes'] + fin['val_bytes'])} of raw text
({fin['train_rows'] + fin['val_rows']:,} documents) across
{fin['train_shards']} train shards and {fin['val_shards']} validation shards.
{stats['removed_ids']:,} documents were removed by dedup/decontamination.

## Schema

| field | type | description |
| --- | --- | --- |
| `id` | string | sha1 of `text` (used for exact dedup) |
| `text` | string | the training text |
| `source` | string | one of `code`, `web`, `math`, `wiki`, `docs` |
| `meta` | string | JSON provenance (language, url, repo, score, ...) |

## Provenance

- Code: [`bigcode/starcoderdata`](https://huggingface.co/datasets/bigcode/starcoderdata).
- Web: [`HuggingFaceFW/fineweb-edu`](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) (`sample-10BT`).
- Math: [`HuggingFaceTB/finemath`](https://huggingface.co/datasets/HuggingFaceTB/finemath) (`finemath-4plus`).
- Wikipedia: [`wikimedia/wikipedia`](https://huggingface.co/datasets/wikimedia/wikipedia) (`20231101.en`).
- Docs: Hub documentation datasets (see notebook docs cell).

## Processing

Per-source quality filtering, PII redaction, global exact dedup, benchmark decontamination,
and bucket-shuffled final corpus. See the notebook for details.
"""


def _ensure_data_mirror(mirror_dir: str, state: dict) -> None:
    if state.get("mirror_complete") and os.path.isdir(mirror_dir):
        if any(Path(mirror_dir).rglob("*.parquet")):
            log.info("[finalize] data mirror present at %s; skipping download.", mirror_dir)
            return
    log.info("[finalize] mirroring data/** to %s ...", mirror_dir)
    os.makedirs(mirror_dir, exist_ok=True)
    snapshot_download(
        repo_id=CONFIG["hf"]["repo_id"],
        repo_type="dataset",
        allow_patterns=["data/**"],
        local_dir=mirror_dir,
        token=HF_TOKEN,
    )
    state["mirror_complete"] = True
    save_finalize_state(state)
    log.info("[finalize] mirror complete.")


def _partition_shard_arrow(
    shard_path: str,
    k: int,
    removed_arr: pa.Array | None,
    shuffle_dir: str,
) -> None:
    table = pq.read_table(shard_path)
    if removed_arr is not None and len(removed_arr) > 0:
        table = table.filter(pc.invert(pc.is_in(table["id"], removed_arr)))
    n = table.num_rows
    if n == 0:
        return
    ids = table.column("id").to_pylist()
    buckets = np.fromiter(
        (zlib.crc32(i.encode()) % k for i in ids), dtype=np.int32, count=n
    )
    shard_tag = os.path.basename(shard_path).replace(".parquet", "")
    for b in np.unique(buckets):
        idx = np.where(buckets == b)[0]
        sub = table.take(pa.array(idx, type=pa.uint64()))
        bdir = os.path.join(shuffle_dir, f"bucket_{int(b):03d}")
        os.makedirs(bdir, exist_ok=True)
        out = os.path.join(bdir, f"part_{shard_tag}.parquet")
        pq.write_table(sub, out, compression="zstd")


def _process_bucket(
    b: int,
    shuffle_dir: str,
    pending_dir: str,
    seed: int,
    val_frac: float,
) -> dict:
    """Shuffle one bucket, split train/val, write pending parquet files."""
    bdir = os.path.join(shuffle_dir, f"bucket_{b:03d}")
    if not os.path.isdir(bdir):
        return {"bucket": b, "per_source": {}, "train_rows": 0, "train_bytes": 0,
                "val_rows": 0, "val_bytes": 0}
    parts = sorted(glob.glob(os.path.join(bdir, "*.parquet")))
    tables = [pq.read_table(p) for p in parts]
    table = pa.concat_tables(tables) if len(tables) > 1 else tables[0]
    n = table.num_rows
    if n == 0:
        return {"bucket": b, "per_source": {}, "train_rows": 0, "train_bytes": 0,
                "val_rows": 0, "val_bytes": 0}

    rng = random.Random(seed + b)
    indices = list(range(n))
    rng.shuffle(indices)
    shuffled = table.take(pa.array(indices, type=pa.uint64()))

    val_flags = pa.array([rng.random() < val_frac for _ in range(n)])
    train_tbl = shuffled.filter(pc.invert(val_flags))
    val_tbl = shuffled.filter(val_flags)

    train_dir = os.path.join(pending_dir, "train")
    val_dir = os.path.join(pending_dir, "validation")
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(val_dir, exist_ok=True)

    train_path = os.path.join(train_dir, f"bucket_{b:03d}.parquet")
    val_path = os.path.join(val_dir, f"bucket_{b:03d}.parquet")
    if train_tbl.num_rows:
        pq.write_table(train_tbl, train_path, compression="zstd", compression_level=9)
    elif os.path.exists(train_path):
        os.remove(train_path)
    if val_tbl.num_rows:
        pq.write_table(val_tbl, val_path, compression="zstd", compression_level=9)
    elif os.path.exists(val_path):
        os.remove(val_path)

    per_source: dict = {}
    for split_tbl in (train_tbl, val_tbl):
        if split_tbl.num_rows == 0:
            continue
        srcs = split_tbl.column("source").to_pylist()
        texts = split_tbl.column("text").to_pylist()
        for src, text in zip(srcs, texts):
            st = per_source.setdefault(src, [0, 0])
            st[0] += 1
            st[1] += len(text.encode("utf-8"))

    return {
        "bucket": b,
        "per_source": per_source,
        "train_rows": train_tbl.num_rows,
        "train_bytes": _table_text_bytes(train_tbl),
        "val_rows": val_tbl.num_rows,
        "val_bytes": _table_text_bytes(val_tbl),
    }


def _merge_pending_split(
    split: str,
    pending_dir: str,
    staging_dir: str,
    target_bytes: int,
    start_index: int,
) -> tuple[list[str], int, int, int]:
    """Merge pending bucket files into ~target_bytes final shards. Returns paths, rows, bytes, next_index."""
    in_dir = os.path.join(pending_dir, split)
    os.makedirs(staging_dir, exist_ok=True)
    bucket_files = sorted(glob.glob(os.path.join(in_dir, "bucket_*.parquet")))
    staged: list[str] = []
    rows = bytes_ = 0
    part_idx = start_index
    accum: list[pa.Table] = []
    accum_bytes = 0

    def flush_accum() -> None:
        nonlocal part_idx, accum, accum_bytes, rows, bytes_
        if not accum:
            return
        out_tbl = pa.concat_tables(accum) if len(accum) > 1 else accum[0]
        base = f"part-{part_idx:05d}.parquet"
        local = os.path.join(staging_dir, base)
        pq.write_table(out_tbl, local, schema=SCHEMA, compression="zstd", compression_level=9)
        staged.append(local)
        rows += out_tbl.num_rows
        bytes_ += _table_text_bytes(out_tbl)
        part_idx += 1
        accum = []
        accum_bytes = 0

    for bf in bucket_files:
        tbl = pq.read_table(bf)
        tbl_bytes = _table_text_bytes(tbl)
        if accum and accum_bytes + tbl_bytes > target_bytes:
            flush_accum()
        accum.append(tbl)
        accum_bytes += tbl_bytes
        if accum_bytes >= target_bytes:
            flush_accum()
    flush_accum()
    return staged, rows, bytes_, part_idx


_UPLOADED_KEYS = {"train": "uploaded_train_parts", "validation": "uploaded_val_parts"}


def _upload_staged_parts(
    staged_paths: list[str],
    split: str,
    state: dict,
    batch_size: int,
) -> list[str]:
    """Batch-upload staged local files sequentially (one commit per batch)."""
    uploaded_key = _UPLOADED_KEYS[split]
    already = set(state.get(uploaded_key, []))
    repo_prefix = f"final/{split}/"
    pending = []
    for local in staged_paths:
        if not os.path.isfile(local):
            continue
        base = os.path.basename(local)
        repo = repo_prefix + base
        if repo in already:
            continue
        pending.append((local, repo))
    if not pending:
        return sorted(already)

    batches = [pending[i:i + batch_size] for i in range(0, len(pending), batch_size)]
    log.info(
        "[finalize] uploading %s %s shards in %s batch commit(s)...",
        len(pending), split, len(batches),
    )
    uploaded_new: list[str] = []
    for i, batch in enumerate(batches):
        upload_batch_with_retry(batch, f"Finalize {split} batch {i + 1}/{len(batches)}")
        uploaded_new.extend(repo for _, repo in batch)
        state[uploaded_key] = sorted(already | set(uploaded_new))
        save_finalize_state(state)

    for local, repo in pending:
        if repo in uploaded_new:
            try:
                os.remove(local)
            except OSError:
                pass
    return state[uploaded_key]


def finalize():
    fcfg = CONFIG["finalize"]
    k = fcfg["shuffle_buckets"]
    target_bytes = fcfg["final_shard_bytes"]
    val_frac = fcfg["val_fraction"]
    batch_size = fcfg.get("upload_batch_size", 16)
    read_workers = fcfg.get("parallel_read_workers", 4)
    bucket_workers = fcfg.get("parallel_bucket_workers", 2)
    seed = CONFIG["seed"]

    scratch = CONFIG["scratch_dir"]
    mirror_dir = os.path.join(scratch, fcfg.get("mirror_subdir", "data_mirror"))
    shuffle_dir = os.path.join(scratch, "shuffle")
    pending_dir = os.path.join(scratch, "pending")
    staging_dir = os.path.join(scratch, "final_staging")

    state = load_finalize_state()
    if state.get("done"):
        log.info("[finalize] already complete per manifest/finalize.json. Skipping.")
        return

    # ---- removed ids -------------------------------------------------------
    removed_arr: pa.Array | None = None
    removed_count = 0
    try:
        rp = hf_hub_download(
            repo_id=CONFIG["hf"]["repo_id"], repo_type="dataset",
            filename="dedup/removed_ids.parquet", token=HF_TOKEN,
        )
        removed_list = pq.read_table(rp, columns=["id"]).column("id").to_pylist()
        removed_count = len(removed_list)
        if removed_list:
            removed_arr = pa.array(removed_list, type=pa.string())
    except Exception:  # noqa: BLE001
        log.warning("No removed_ids.parquet found; finalizing without removals.")
    log.info("[finalize] applying %s removed ids.", removed_count)

    # ---- mirror ------------------------------------------------------------
    _ensure_data_mirror(mirror_dir, state)

    # ---- phase 1: arrow partition ------------------------------------------
    os.makedirs(shuffle_dir, exist_ok=True)
    partitioned = set(state.get("partitioned_sources", []))

    for source in SOURCES:
        if source in partitioned:
            log.info("[finalize] phase1 skip (already partitioned): %s", source)
            continue
        shards = list_source_shards(source)
        if not shards:
            partitioned.add(source)
            state["partitioned_sources"] = sorted(partitioned)
            save_finalize_state(state)
            continue
        local_shards = [os.path.join(mirror_dir, s) for s in shards]
        missing = [p for p in local_shards if not os.path.isfile(p)]
        if missing:
            raise FileNotFoundError(f"Missing mirrored shards ({len(missing)}); re-run mirror: {missing[:3]}")

        log.info("[finalize] phase1 partitioning %s (%s shards, %s workers)...",
                 source, len(local_shards), read_workers)
        with ThreadPoolExecutor(max_workers=read_workers) as ex:
            futs = [
                ex.submit(
                    _partition_shard_arrow,
                    path, k, removed_arr, shuffle_dir,
                )
                for path in local_shards
            ]
            for fut in as_completed(futs):
                fut.result()

        partitioned.add(source)
        state["partitioned_sources"] = sorted(partitioned)
        save_finalize_state(state)
        log.info("[finalize] phase1 done: %s", source)

    if state.get("mirror_complete") and os.path.isdir(mirror_dir):
        log.info("[finalize] removing data mirror to free disk...")
        shutil.rmtree(mirror_dir, ignore_errors=True)

    # ---- phase 2: parallel bucket shuffle + train/val split ----------------
    os.makedirs(pending_dir, exist_ok=True)
    buckets_done = set(state.get("buckets_done", []))
    per_source_final = state.get("per_source_final", {})
    train_rows = state.get("train_rows", 0)
    val_rows = state.get("val_rows", 0)
    train_bytes = state.get("train_bytes", 0)
    val_bytes = state.get("val_bytes", 0)

    todo_buckets = [b for b in range(k) if b not in buckets_done]
    if todo_buckets:
        log.info("[finalize] phase2 processing %s buckets (%s workers)...",
                 len(todo_buckets), bucket_workers)
        with ThreadPoolExecutor(max_workers=bucket_workers) as ex:
            futs = {
                ex.submit(_process_bucket, b, shuffle_dir, pending_dir, seed, val_frac): b
                for b in todo_buckets
            }
            for fut in as_completed(futs):
                res = fut.result()
                b = res["bucket"]
                buckets_done.add(b)
                train_rows += res["train_rows"]
                val_rows += res["val_rows"]
                train_bytes += res["train_bytes"]
                val_bytes += res["val_bytes"]
                for src, (r, by) in res["per_source"].items():
                    cur = per_source_final.setdefault(src, {"rows": 0, "bytes": 0})
                    cur["rows"] += r
                    cur["bytes"] += by
                bdir = os.path.join(shuffle_dir, f"bucket_{b:03d}")
                shutil.rmtree(bdir, ignore_errors=True)
                state["buckets_done"] = sorted(buckets_done)
                state["per_source_final"] = per_source_final
                state["train_rows"] = train_rows
                state["val_rows"] = val_rows
                state["train_bytes"] = train_bytes
                state["val_bytes"] = val_bytes
                save_finalize_state(state)
                if len(buckets_done) % 8 == 0 or len(buckets_done) == k:
                    log.info("[finalize] phase2 buckets done: %s/%s", len(buckets_done), k)

    shutil.rmtree(shuffle_dir, ignore_errors=True)

    if len(buckets_done) == k and not state.get("staged_train_parts"):
        pending_any = any(Path(pending_dir).rglob("bucket_*.parquet"))
        if not pending_any:
            raise RuntimeError(
                "[finalize] manifest says buckets are done but local pending data is missing. "
                "If this is a new Colab session, delete manifest/finalize.json on the Hub "
                "(or clear buckets_done in it) and re-run finalize from phase 1."
            )

    # ---- phase 3: merge pending -> staged final shards ---------------------

    staged_train: list[str] = []
    staged_val: list[str] = []
    train_stage_dir = os.path.join(staging_dir, "train")
    val_stage_dir = os.path.join(staging_dir, "validation")

    staged_train = [
        os.path.join(train_stage_dir, p) for p in state.get("staged_train_parts", [])
        if os.path.isfile(os.path.join(train_stage_dir, p))
    ]
    if not staged_train:
        log.info("[finalize] phase3 merging train pending shards...")
        staged_train, tr, tb, _ = _merge_pending_split(
            "train", pending_dir, train_stage_dir, target_bytes, 0,
        )
        state["staged_train_parts"] = [os.path.basename(p) for p in staged_train]
        state["train_rows"] = tr
        state["train_bytes"] = tb
        save_finalize_state(state)

    staged_val = [
        os.path.join(val_stage_dir, p) for p in state.get("staged_val_parts", [])
        if os.path.isfile(os.path.join(val_stage_dir, p))
    ]
    if not staged_val:
        log.info("[finalize] phase3 merging validation pending shards...")
        staged_val, vr, vb, _ = _merge_pending_split(
            "validation", pending_dir, val_stage_dir, target_bytes, 0,
        )
        state["staged_val_parts"] = [os.path.basename(p) for p in staged_val]
        state["val_rows"] = vr
        state["val_bytes"] = vb
        save_finalize_state(state)

    shutil.rmtree(pending_dir, ignore_errors=True)

    # ---- phase 4: batch upload ---------------------------------------------
    uploaded_train = _upload_staged_parts(staged_train, "train", state, batch_size)
    uploaded_val = _upload_staged_parts(staged_val, "validation", state, batch_size)

    # ---- stats + dataset card ----------------------------------------------
    stats = {
        "final": {
            "train_rows": state["train_rows"],
            "train_bytes": state["train_bytes"],
            "train_shards": len(uploaded_train),
            "val_rows": state["val_rows"],
            "val_bytes": state["val_bytes"],
            "val_shards": len(uploaded_val),
        },
        "per_source_final": per_source_final,
        "per_source_collected": {},
        "removed_ids": removed_count,
    }
    for source in SOURCES:
        st = load_state(source)
        if st.get("sub"):
            rows = sum(u["rows_written"] for u in st["sub"].values())
            byts = sum(u["bytes_written"] for u in st["sub"].values())
        else:
            rows, byts = st.get("rows_written", 0), st.get("bytes_written", 0)
        stats["per_source_collected"][source] = {"rows": rows, "bytes": byts}

    stats_local = os.path.join(scratch, "stats.json")
    with open(stats_local, "wb") as fh:
        fh.write(orjson.dumps(stats, option=orjson.OPT_INDENT_2))
    upload_with_retry(stats_local, "stats.json")
    os.remove(stats_local)

    card_local = os.path.join(scratch, "README.md")
    with open(card_local, "w", encoding="utf-8") as fh:
        fh.write(_write_dataset_card(stats))
    upload_with_retry(card_local, "README.md")
    os.remove(card_local)

    shutil.rmtree(staging_dir, ignore_errors=True)

    state["done"] = True
    save_finalize_state(state)

    total = stats["final"]["train_bytes"] + stats["final"]["val_bytes"]
    log.info(
        "=== Finalize complete: %s train + %s val = %s total (%s train shards) ===",
        f"{stats['final']['train_rows']:,}",
        f"{stats['final']['val_rows']:,}",
        _fmt_gb(total),
        stats["final"]["train_shards"],
    )
    log.info("Dataset: https://huggingface.co/datasets/%s", CONFIG["hf"]["repo_id"])


finalize()

## 11. Sanity checks

Reads the final corpus back from the Hub: prints `stats.json`, verifies per-source budgets, and streams a few sample documents from the `train` split so you can eyeball quality before moving on to tokenizer/training work.

In [ ]:
def sanity_check(n_samples: int = 3):
    # 1) Print the stats report.
    try:
        sp = hf_hub_download(repo_id=CONFIG["hf"]["repo_id"], repo_type="dataset",
                             filename="stats.json", token=HF_TOKEN, force_download=True)
        with open(sp, "rb") as fh:
            stats = json.loads(fh.read())
        print("=== stats.json ===")
        print(json.dumps(stats, indent=2))
        final = stats["final"]
        total_gb = (final["train_bytes"] + final["val_bytes"]) / GB
        print(f"\nTotal final corpus: {total_gb:.2f} GB "
              f"({final['train_rows'] + final['val_rows']:,} docs)")
    except Exception as exc:  # noqa: BLE001
        log.warning("Could not read stats.json (did finalize run?): %s", exc)

    # 2) Stream a few train samples back from the Hub.
    print(f"\n=== {n_samples} sample train documents ===")
    try:
        ds = load_dataset(CONFIG["hf"]["repo_id"], split="train", streaming=True, token=HF_TOKEN)
        for i, row in enumerate(ds):
            if i >= n_samples:
                break
            preview = row["text"][:500].replace("\n", " ")
            print(f"\n[{i}] source={row['source']} id={row['id'][:12]} meta={row['meta'][:120]}")
            print(f"    {preview}")
    except Exception as exc:  # noqa: BLE001
        log.warning("Could not stream final dataset: %s", exc)


sanity_check()

## Next steps (out of scope here)

The corpus now lives at `datasets/<user>/coder-pretrain-60gb` with `train` / `validation` splits, a `stats.json`, and a dataset card. From here, the separate training phase would:

1. Train a byte-level BPE tokenizer on a sample of this corpus (or reuse an existing coder tokenizer).
2. Pre-tokenize and pack into fixed-length sequences.
3. Pre-train the 1-5B model (streamed from the Hub) on A100/H100.
4. SFT on `nvidia/OpenCodeInstruct`.

**Re-running / resuming:** every source cell is idempotent and resumes from the Hub manifest, so after a Colab disconnect just re-run the cells top to bottom - completed shards are skipped. **Finalize** resumes via `manifest/finalize.json` (mirror → partition → buckets → batch upload). To rebuild a source from scratch, delete its `manifest/<source>.json` and `data/<source>/` on the Hub first.